# Huấn Luyện Mô Hình Đề Xuất Cuối Cùng (Exp_G: DACBAM + ASL + CWT)

**Đóng góp chính (Contributions):**
1. **EfficientNet-B0:** Nhẹ hơn ResNet50 nhưng khả năng trích xuất đặc trưng cực tốt.
2. **Difficulty-Aware CBAM (DACBAM):** Tích hợp Masked Attention Regularization (mask_prob=0.3). Trong quá trình train, cố tình che đi các vùng đặc trưng dễ (easy regions) để ép CBAM tập trung vào vùng khó (hard regions/nhiễu), giải quyết triệt để tình trạng CBAM gây overfit.
3. **ASL + Class-wise Threshold (CWT):** ASL điều khiển hàm Loss để chống lệch dữ liệu lúc train. CWT tự động tìm ngưỡng xác suất tối ưu cho từng class riêng biệt để chống lệch dữ liệu lúc Inference (Đã tích hợp trong `evaluate.py`).

In [ ]:
!pip install timm pycocotools torchmetrics scikit-learn pyyaml -q
import os
from pathlib import Path

REPO_URL = 'https://github.com/Thinh59/ECAAL.git'
REPO_DIR = Path('/kaggle/working/ECAAL')

if REPO_DIR.exists():
    os.system(f'git -C {REPO_DIR} pull')
else:
    os.system(f'git clone {REPO_URL} {REPO_DIR}')

In [ ]:
# Copy thư mục chia dữ liệu đã có sẵn từ Dataset để đảm bảo train/val đồng nhất
import shutil
SRC_SUBSET = Path('/kaggle/input/datasets/thinhha59/models/results/data/coco_subset')
SUBSET_DIR = Path('/kaggle/working/data/coco_subset')
SUBSET_DIR.mkdir(parents=True, exist_ok=True)
if SRC_SUBSET.exists():
    for fname in ['subset_train_ids.json', 'subset_val_ids.json', 'subset_test_ids.json']:
        if (SRC_SUBSET / fname).exists():
            shutil.copy2(SRC_SUBSET / fname, SUBSET_DIR / fname)
            print(f'Copied {fname}')

In [ ]:
# BẮT ĐẦU HUẤN LUYỆN MODEL ĐỀ XUẤT CUỐI CÙNG
import sys
sys.path.insert(0, str(REPO_DIR / 'src'))
config_path = str(REPO_DIR / 'configs' / 'exp_G_efficientnet_dacbam_asl_cwt.yaml')

!python {REPO_DIR}/src/train.py --config {config_path}